In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ai-tudy/finale/pipeline_detected_family_image

Mounted at /content/drive
/content/drive/MyDrive/ai-tudy/finale/pipeline_detected_family_image


In [2]:
import sys
sys.path.insert(0, './utils/')
from dataset import ConvDataset
from trainer import train_model, get_y_true_pred
from other_utils import get_model_resnet18, view_classification_report, load_model, save_json
from downloading_from_blur_dataset import download_images_blur_dataset
from PIL import Image
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import os
from tqdm import tqdm
from copy import deepcopy
from sklearn.metrics import f1_score, classification_report




Скачивание изображений с датасета blur-dataset

In [3]:
download_images_blur_dataset()

Using Colab cache for faster access to the 'blur-dataset' dataset.
Датасет скачан в: /kaggle/input/blur-dataset
Целевые папки:
  Train blur: data/train/blur_dataset/blur
  Train sharp: data/train/blur_dataset/sharp
  Val blur: data/val/blur_dataset/blur
  Val sharp: data/val/blur_dataset/sharp
Скопировано в train/blur: 250
Скопировано в train/sharp: 250
Скопировано в val/blur: 100
Скопировано в val/sharp: 100


Основная работа по обучении модели.

In [3]:
root_dir_train = 'data/train/blur_dataset'
root_dir_val = 'data/val/blur_dataset'
dataset_train = ConvDataset(root_dir_train)
dataset_val = ConvDataset(root_dir_val)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(dataset_train, batch_size=16, shuffle=True, num_workers=2, pin_memory=True, prefetch_factor=2)
val_loader = DataLoader(dataset_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True, prefetch_factor=2)

In [ ]:
dataset_train.dict_class_label

In [7]:
model = get_model_resnet18().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=7,
    eta_min=1e-6
)

In [8]:
path_save = 'models/blur_models/best_model_blur.pth'
history = train_model(model, train_loader, val_loader, path_save, criterion, optimizer, scheduler, device=device)



Epoch 1/15


Test: 100%|██████████| 13/13 [00:10<00:00,  1.26it/s]


Train Loss: 0.5699 | Train Acc: 0.8180
Val   Loss: 0.5997 | Val   Acc: 0.7200
--------------------
--------------------

Epoch 2/15


Test: 100%|██████████| 13/13 [00:08<00:00,  1.47it/s]


Train Loss: 0.3141 | Train Acc: 0.8600
Val   Loss: 0.1364 | Val   Acc: 0.9500
--------------------
--------------------

Epoch 3/15


...

++++++++++++++++++++++++++++
Early Stopping: 3 / 5 эпох без улучшений. (Лучший лосс: 0.0659945809841156, Текущий лосс: 0.24551930889487267)
++++++++++++++++++++++++++++
Train Loss: 0.1509 | Train Acc: 0.9460
Val   Loss: 0.2455 | Val   Acc: 0.8900
--------------------
--------------------

Epoch 14/15


Test: 100%|██████████| 13/13 [00:07<00:00,  1.84it/s]


++++++++++++++++++++++++++++
Early Stopping: 4 / 5 эпох без улучшений. (Лучший лосс: 0.0659945809841156, Текущий лосс: 0.25906538728624584)
++++++++++++++++++++++++++++
Train Loss: 0.1506 | Train Acc: 0.9460
Val   Loss: 0.2591 | Val   Acc: 0.9100
--------------------
--------------------

Epoch 15/15


Test: 100%|██████████| 13/13 [00:06<00:00,  2.01it/s]

++++++++++++++++++++++++++++
Early Stopping: 5 / 5 эпох без улучшений. (Лучший лосс: 0.0659945809841156, Текущий лосс: 0.2399575488269329)
++++++++++++++++++++++++++++
Train Loss: 0.0769 | Train Acc: 0.9740
Val   Loss: 0.2400 | Val   Acc: 0.9050
--------------------
Сработала рання остановка!


Тестирование

In [9]:
model = load_model('models/blur_models/best_model_blur.pth')

In [10]:
root_dir_test = 'data/test/blur_dataset'
dataset_test = ConvDataset(root_dir_test)
test_loader = DataLoader(dataset_test, batch_size=4, shuffle=False)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
y_true, y_pred = get_y_true_pred(model, test_loader, device)

Test: 100%|██████████| 20/20 [00:18<00:00,  1.10it/s]


In [12]:
view_classification_report(y_true, y_pred, dataset_test.class_name_list)

              precision    recall  f1-score   support

        blur       0.94      0.82      0.88        40
       sharp       0.84      0.95      0.89        40

    accuracy                           0.89        80
   macro avg       0.89      0.89      0.89        80
weighted avg       0.89      0.89      0.89        80



In [13]:
save_json(dataset_test.dict_class_label, 'labels/label_blur.json')

JSON успешно сохранен по пути: labels/label_blur.json
